In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.pipeline import Pipeline
from scipy.stats import randint, uniform
import joblib
import time

# 1. Load and prepare data
print("Loading and preparing data...")
df = pd.read_csv('balanced_datasets/multi_output_balanced.csv')

# Separate features and targets
X = df.drop(columns=['oestrus', 'calving', 'lameness', 'mastitis', 'cow', 'date', 'any_disease'])
y = df[['oestrus', 'calving', 'lameness', 'mastitis']]

# Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y['oestrus']
)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

# 2. Create pipeline with scaling and model
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', MultiOutputClassifier(
        RandomForestClassifier(
            class_weight='balanced_subsample',
            random_state=42,
            n_jobs=-1
        )
    ))
])

# 3. Hyperparameter tuning setup
param_dist = {
    'clf__estimator__n_estimators': randint(100, 500),
    'clf__estimator__max_depth': [10, 20, 30, None],
    'clf__estimator__min_samples_split': randint(2, 20),
    'clf__estimator__min_samples_leaf': randint(1, 10),
    'clf__estimator__max_features': ['sqrt', 'log2', 0.5],
    'clf__estimator__bootstrap': [True, False],
    'clf__estimator__max_samples': uniform(0.5, 0.5)
}

# Custom cross-validation for multi-output stratification
def multilabel_stratified_cv(X, y, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    dummy = y.iloc[:, 0]  # Use first label for stratification
    return skf.split(X, dummy)

# 4. Randomized Search with Cross-Validation
print("Starting hyperparameter tuning with RandomizedSearchCV...")
start_time = time.time()

search = RandomizedSearchCV(
    pipeline,
    param_dist,
    n_iter=50,
    cv=multilabel_stratified_cv(X_train, y_train, n_splits=3),
    scoring='roc_auc_ovr_weighted',
    refit=True,
    random_state=42,
    verbose=3,
    n_jobs=-1
)

search.fit(X_train, y_train)

print(f"Hyperparameter tuning completed in {time.time()-start_time:.2f} seconds")
print(f"Best parameters: {search.best_params_}")
print(f"Best CV score: {search.best_score_:.4f}")

# 5. Evaluate on test set
print("\nEvaluating on test set...")
y_pred = search.predict(X_test)
y_proba = search.predict_proba(X_test)  # List of arrays for each disease

# Classification report for each disease
diseases = ['oestrus', 'calving', 'lameness', 'mastitis']
test_results = {}

for i, disease in enumerate(diseases):
    print(f"\n{'='*50}\n{disease.upper()} PERFORMANCE\n{'='*50}")
    print(classification_report(y_test[disease], y_pred[:, i]))
    
    # Store metrics
    test_results[disease] = {
        'auc': roc_auc_score(y_test[disease], y_proba[i][:, 1])
    }

# 6. Feature Importance Analysis (for all diseases)
print("\nFeature Importances:")
best_model = search.best_estimator_.named_steps['clf']
feature_importances = []

# Collect importances for each disease
for idx, disease in enumerate(diseases):
    importances = best_model.estimators_[idx].feature_importances_
    feature_importances.append(importances)

# Create importance dataframe
importance_df = pd.DataFrame(
    np.array(feature_importances).T,
    columns=diseases,
    index=X.columns
)

# Add average importance column
importance_df['Average'] = importance_df.mean(axis=1)
importance_df = importance_df.sort_values('Average', ascending=False)

print("\nTop 10 Features:")
print(importance_df.head(10))

# 7. Save the best model and results
joblib.dump(search.best_estimator_, 'random_forest_multioutput_model.joblib')
importance_df.to_csv('feature_importances.csv')
print("Model and feature importances saved")

# 8. Final summary
print("\n" + "="*50)
print("FINAL MODEL SUMMARY")
print("="*50)
print(f"Best parameters: {search.best_params_}")
print(f"Cross-validation AUC: {search.best_score_:.4f}")

for disease in diseases:
    print(f"{disease.upper()} Test AUC: {test_results[disease]['auc']:.4f}")

Loading and preparing data...
Train shape: (26304, 12), Test shape: (6576, 12)
Starting hyperparameter tuning with RandomizedSearchCV...
Fitting 3 folds for each of 50 candidates, totalling 150 fits


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
72 fits failed out of a total of 150.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
72 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 473, in fit
    self._final_estimator.fi

Hyperparameter tuning completed in 349.23 seconds
Best parameters: {'clf__estimator__bootstrap': True, 'clf__estimator__max_depth': 30, 'clf__estimator__max_features': 0.5, 'clf__estimator__max_samples': 0.9281621459390462, 'clf__estimator__min_samples_leaf': 6, 'clf__estimator__min_samples_split': 2, 'clf__estimator__n_estimators': 215}
Best CV score: 0.9275

Evaluating on test set...

OESTRUS PERFORMANCE
              precision    recall  f1-score   support

           0       0.94      0.89      0.92      4987
           1       0.72      0.83      0.77      1589

    accuracy                           0.88      6576
   macro avg       0.83      0.86      0.84      6576
weighted avg       0.89      0.88      0.88      6576


CALVING PERFORMANCE
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      5668
           1       0.65      0.64      0.65       908

    accuracy                           0.90      6576
   macro avg       0.80 

In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.pipeline import Pipeline
from scipy.stats import randint, uniform
import joblib
import time

# 1. Load and prepare data
print("Loading and preparing data...")
df = pd.read_csv('balanced_datasets/multi_output_balanced.csv')

# Remove problematic features
df = df.drop(columns=['Unnamed: 0', 'Unnamed: 1', 'disturbance'])

# Separate features and targets
X = df.drop(columns=['oestrus', 'calving', 'lameness', 'mastitis', 'cow', 'date', 'any_disease'])
y = df[['oestrus', 'calving', 'lameness', 'mastitis']]

# Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y['oestrus']
)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Features used: {list(X.columns)}")

# 2. Create pipeline with scaling and model
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', MultiOutputClassifier(
        RandomForestClassifier(
            class_weight='balanced_subsample',
            random_state=42,
            n_jobs=-1
        )
    ))
])

# 3. Hyperparameter tuning setup with conditional parameters
param_dist = {
    'clf__estimator__n_estimators': randint(100, 500),
    'clf__estimator__max_depth': [10, 20, 30, None],
    'clf__estimator__min_samples_split': randint(2, 20),
    'clf__estimator__min_samples_leaf': randint(1, 10),
    'clf__estimator__max_features': ['sqrt', 'log2', 0.5],
    'clf__estimator__bootstrap': [True, False],
    # max_samples will be handled conditionally
    'clf__estimator__max_samples': uniform(0.5, 0.5)
}

# Custom cross-validation for multi-output stratification
def multilabel_stratified_cv(X, y, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    dummy = y.iloc[:, 0]
    return skf.split(X, dummy)

# 4. Randomized Search with Cross-Validation
print("Starting hyperparameter tuning with RandomizedSearchCV...")
start_time = time.time()

# Create custom parameter sampler
def custom_param_sampler(param_dist, n_iter=10, random_state=None):
    rng = np.random.RandomState(random_state)
    samples = []
    
    for _ in range(n_iter):
        params = {}
        # First sample bootstrap
        bootstrap = rng.choice(param_dist['clf__estimator__bootstrap'])
        params['clf__estimator__bootstrap'] = bootstrap
        
        # Handle max_samples based on bootstrap
        if bootstrap:
            # Sample max_samples only when bootstrap is True
            params['clf__estimator__max_samples'] = param_dist['clf__estimator__max_samples'].rvs(random_state=rng)
        else:
            # Set max_samples to None when bootstrap is False
            params['clf__estimator__max_samples'] = None
        
        # Sample other parameters
        for key in param_dist:
            if key in ['clf__estimator__bootstrap', 'clf__estimator__max_samples']:
                continue  # Already handled
                
            if hasattr(param_dist[key], 'rvs'):
                params[key] = param_dist[key].rvs(random_state=rng)
            elif isinstance(param_dist[key], list):
                params[key] = rng.choice(param_dist[key])
            else:
                params[key] = param_dist[key]
                
        samples.append(params)
    return samples

# Generate custom parameter combinations
custom_params = custom_param_sampler(param_dist, n_iter=50, random_state=42)

# Create a custom search CV class
class CustomRandomSearch(RandomizedSearchCV):
    def _run_search(self, evaluate_candidates):
        evaluate_candidates(custom_params)

# 5. Create and run the custom search
search = CustomRandomSearch(
    pipeline,
    param_dist,  # Still need to pass this for initialization, but we override _run_search
    cv=multilabel_stratified_cv(X_train, y_train, n_splits=3),
    scoring='roc_auc_ovr_weighted',
    refit=True,
    random_state=42,
    verbose=3,
    n_jobs=-1
)

search.fit(X_train, y_train)

print(f"Hyperparameter tuning completed in {time.time()-start_time:.2f} seconds")
print(f"Best parameters: {search.best_params_}")
print(f"Best CV score: {search.best_score_:.4f}")

# 6. Evaluate on test set
print("\nEvaluating on test set...")
y_pred = search.predict(X_test)
y_proba = search.predict_proba(X_test)

# Classification report for each disease
diseases = ['oestrus', 'calving', 'lameness', 'mastitis']
test_results = {}

for i, disease in enumerate(diseases):
    print(f"\n{'='*50}\n{disease.upper()} PERFORMANCE\n{'='*50}")
    print(classification_report(y_test[disease], y_pred[:, i]))
    test_results[disease] = {
        'auc': roc_auc_score(y_test[disease], y_proba[i][:, 1])
    }

# 7. Feature Importance Analysis
print("\nFeature Importances:")
best_model = search.best_estimator_.named_steps['clf']
feature_importances = []

for idx, disease in enumerate(diseases):
    importances = best_model.estimators_[idx].feature_importances_
    feature_importances.append(importances)

importance_df = pd.DataFrame(
    np.array(feature_importances).T,
    columns=diseases,
    index=X.columns
)
importance_df['Average'] = importance_df.mean(axis=1)
importance_df = importance_df.sort_values('Average', ascending=False)

print("\nTop Features:")
print(importance_df)

# 8. Save the best model and results
joblib.dump(search.best_estimator_, 'random_forest_multioutput_model.joblib')
importance_df.to_csv('feature_importances.csv')
print("Model and feature importances saved")

# 9. Final summary
print("\n" + "="*50)
print("FINAL MODEL SUMMARY")
print("="*50)
print(f"Best parameters: {search.best_params_}")
print(f"Cross-validation AUC: {search.best_score_:.4f}")

for disease in diseases:
    print(f"{disease.upper()} Test AUC: {test_results[disease]['auc']:.4f}")

Loading and preparing data...
Train shape: (26304, 9), Test shape: (6576, 9)
Features used: ['cow.1', 'hour', 'IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'rest_to_eat_ratio', 'hour_sin', 'hour_cos']
Starting hyperparameter tuning with RandomizedSearchCV...
Fitting 3 folds for each of 50 candidates, totalling 150 fits


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
54 fits failed out of a total of 150.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
31 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 473, in fit
    self._final_estimator.fi

Hyperparameter tuning completed in 567.39 seconds
Best parameters: {'clf__estimator__bootstrap': True, 'clf__estimator__max_samples': 0.5025307919231093, 'clf__estimator__n_estimators': 483, 'clf__estimator__max_depth': 20, 'clf__estimator__min_samples_split': 19, 'clf__estimator__min_samples_leaf': 2, 'clf__estimator__max_features': 'log2'}
Best CV score: 0.7889

Evaluating on test set...

OESTRUS PERFORMANCE
              precision    recall  f1-score   support

           0       0.84      0.83      0.84      4987
           1       0.49      0.50      0.50      1589

    accuracy                           0.75      6576
   macro avg       0.67      0.67      0.67      6576
weighted avg       0.76      0.75      0.76      6576


CALVING PERFORMANCE
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      5668
           1       0.63      0.59      0.61       908

    accuracy                           0.90      6576
   macro avg       0

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.pipeline import Pipeline
from scipy.stats import randint, uniform
import joblib
import time

# 1. Load and prepare data
print("Loading and preparing data...")
df = pd.read_csv('balanced_datasets/multi_output_balanced.csv')

# Remove problematic features
df = df.drop(columns=['Unnamed: 0', 'Unnamed: 1', 'disturbance'])

# Separate features and targets
X = df.drop(columns=['oestrus', 'calving', 'lameness', 'mastitis', 'cow', 'date', 'any_disease'])
y = df[['oestrus', 'calving', 'lameness', 'mastitis']]

# Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y['oestrus']
)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Features used: {list(X.columns)}")

# 2. Create pipeline with scaling and model
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', MultiOutputClassifier(
        RandomForestClassifier(
            class_weight='balanced_subsample',
            random_state=42,
            n_jobs=-1
        )
    ))
])

# 3. Hyperparameter tuning setup with validated parameters
param_dist = {
    'clf__estimator__n_estimators': randint(100, 500),
    'clf__estimator__max_depth': [10, 20, 30, None],
    'clf__estimator__min_samples_split': randint(2, 20),
    'clf__estimator__min_samples_leaf': randint(1, 10),
    'clf__estimator__max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7],  # Mixed types handled properly
    'clf__estimator__bootstrap': [True, False],
    'clf__estimator__max_samples': uniform(0.5, 0.5)
}

# Custom cross-validation for multi-output stratification
def multilabel_stratified_cv(X, y, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    dummy = y.iloc[:, 0]
    return skf.split(X, dummy)

# 4. Custom parameter sampler with type safety
def custom_param_sampler(param_dist, n_iter=50, random_state=None):
    rng = np.random.RandomState(random_state)
    samples = []
    
    for _ in range(n_iter):
        params = {}
        # First sample bootstrap
        bootstrap = rng.choice(param_dist['clf__estimator__bootstrap'])
        params['clf__estimator__bootstrap'] = bootstrap
        
        # Handle max_samples based on bootstrap
        if bootstrap:
            params['clf__estimator__max_samples'] = param_dist['clf__estimator__max_samples'].rvs(random_state=rng)
        else:
            params['clf__estimator__max_samples'] = None
        
        # Sample other parameters with type checking
        for key in param_dist:
            if key in ['clf__estimator__bootstrap', 'clf__estimator__max_samples']:
                continue
                
            dist = param_dist[key]
            if key == 'clf__estimator__max_features':
                # Ensure proper type handling for max_features
                if isinstance(dist, list):
                    choice = dist[rng.randint(len(dist))]
                    # Convert to float if numeric string
                    if isinstance(choice, str) and choice.replace('.', '', 1).isdigit():
                        params[key] = float(choice)
                    else:
                        params[key] = choice
                else:
                    params[key] = dist.rvs(random_state=rng)
            else:
                if hasattr(dist, 'rvs'):
                    params[key] = dist.rvs(random_state=rng)
                elif isinstance(dist, list):
                    params[key] = dist[rng.randint(len(dist))]
                else:
                    params[key] = dist
                    
        samples.append(params)
    return samples

# 5. Create and run the search
print("Starting hyperparameter tuning with RandomizedSearchCV...")
start_time = time.time()

# Generate valid parameter combinations
custom_params = custom_param_sampler(param_dist, n_iter=50, random_state=42)

# Use custom parameters directly
search = RandomizedSearchCV(
    pipeline,
    param_distributions=custom_params,
    cv=multilabel_stratified_cv(X_train, y_train, n_splits=3),
    scoring='roc_auc_ovr_weighted',
    refit=True,
    random_state=42,
    verbose=3,
    n_jobs=-1,
    error_score='raise'  # Get immediate feedback on failures
)

search.fit(X_train, y_train)

print(f"Hyperparameter tuning completed in {time.time()-start_time:.2f} seconds")
print(f"Best parameters: {search.best_params_}")
print(f"Best CV score: {search.best_score_:.4f}")

# 6. Evaluate on test set
print("\nEvaluating on test set...")
y_pred = search.predict(X_test)
y_proba = search.predict_proba(X_test)

# Classification report for each disease
diseases = ['oestrus', 'calving', 'lameness', 'mastitis']
test_results = {}

for i, disease in enumerate(diseases):
    print(f"\n{'='*50}\n{disease.upper()} PERFORMANCE\n{'='*50}")
    print(classification_report(y_test[disease], y_pred[:, i]))
    test_results[disease] = {
        'auc': roc_auc_score(y_test[disease], y_proba[i][:, 1])
    }

# 7. Feature Importance Analysis
print("\nFeature Importances:")
best_model = search.best_estimator_.named_steps['clf']
feature_importances = []

for idx, disease in enumerate(diseases):
    importances = best_model.estimators_[idx].feature_importances_
    feature_importances.append(importances)

importance_df = pd.DataFrame(
    np.array(feature_importances).T,
    columns=diseases,
    index=X.columns
)
importance_df['Average'] = importance_df.mean(axis=1)
importance_df = importance_df.sort_values('Average', ascending=False)

print("\nTop Features:")
print(importance_df)

# 8. Save the best model and results
joblib.dump(search.best_estimator_, 'random_forest_multioutput_model.joblib')
importance_df.to_csv('feature_importances.csv')
print("Model and feature importances saved")

# 9. Final summary
print("\n" + "="*50)
print("FINAL MODEL SUMMARY")
print("="*50)
print(f"Best parameters: {search.best_params_}")
print(f"Cross-validation AUC: {search.best_score_:.4f}")

for disease in diseases:
    print(f"{disease.upper()} Test AUC: {test_results[disease]['auc']:.4f}")

Loading and preparing data...
Train shape: (26304, 9), Test shape: (6576, 9)
Features used: ['cow.1', 'hour', 'IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'rest_to_eat_ratio', 'hour_sin', 'hour_cos']
Starting hyperparameter tuning with RandomizedSearchCV...


TypeError: Parameter grid for parameter 'clf__estimator__bootstrap' is not iterable or a distribution (value=True)

In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.pipeline import Pipeline
import joblib
import time

# 1. Load and prepare data
print("Loading and preparing data...")
df = pd.read_csv('balanced_datasets/multi_output_balanced.csv')

# Remove problematic features
df = df.drop(columns=['Unnamed: 0', 'Unnamed: 1', 'disturbance'])

# Separate features and targets
X = df.drop(columns=['oestrus', 'calving', 'lameness', 'mastitis', 'cow', 'date', 'any_disease'])
y = df[['oestrus', 'calving', 'lameness', 'mastitis']]

# Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y['oestrus']
)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Features used: {list(X.columns)}")

# 2. Create pipeline with scaling and model
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', MultiOutputClassifier(
        RandomForestClassifier(
            class_weight='balanced_subsample',
            random_state=42,
            n_jobs=-1
        )
    ))
])

# 3. Manually define parameter combinations to avoid issues
param_combinations = [
    # Best parameters from previous run
    {
        'clf__estimator__bootstrap': [True],
        'clf__estimator__max_samples': [0.5],
        'clf__estimator__n_estimators': [483],
        'clf__estimator__max_depth': [20],
        'clf__estimator__min_samples_split': [19],
        'clf__estimator__min_samples_leaf': [2],
        'clf__estimator__max_features': ['log2']
    },
    
    # Alternative configurations
    {
        'clf__estimator__bootstrap': [True],
        'clf__estimator__max_samples': [0.7],
        'clf__estimator__n_estimators': [300],
        'clf__estimator__max_depth': [30],
        'clf__estimator__min_samples_split': [10],
        'clf__estimator__min_samples_leaf': [4],
        'clf__estimator__max_features': ['sqrt']
    },
    
    {
        'clf__estimator__bootstrap': [True],
        'clf__estimator__max_samples': [0.6],
        'clf__estimator__n_estimators': [200],
        'clf__estimator__max_depth': [None],
        'clf__estimator__min_samples_split': [5],
        'clf__estimator__min_samples_leaf': [1],
        'clf__estimator__max_features': [0.5]  # Float value is acceptable
    },
    
    {
        'clf__estimator__bootstrap': [False],
        'clf__estimator__max_samples': [None],
        'clf__estimator__n_estimators': [400],
        'clf__estimator__max_depth': [20],
        'clf__estimator__min_samples_split': [15],
        'clf__estimator__min_samples_leaf': [2],
        'clf__estimator__max_features': ['log2']
    },
    
    {
        'clf__estimator__bootstrap': [True],
        'clf__estimator__max_samples': [0.8],
        'clf__estimator__n_estimators': [250],
        'clf__estimator__max_depth': [15],
        'clf__estimator__min_samples_split': [8],
        'clf__estimator__min_samples_leaf': [3],
        'clf__estimator__max_features': [0.3]  # Float value is acceptable
    }
]

# 4. Custom cross-validation for multi-output stratification
def multilabel_stratified_cv(X, y, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    dummy = y.iloc[:, 0]
    return skf.split(X, dummy)

# 5. Create and run the search using GridSearchCV
print("Starting hyperparameter tuning with GridSearchCV...")
start_time = time.time()

search = GridSearchCV(
    pipeline,
    param_grid=param_combinations,
    cv=multilabel_stratified_cv(X_train, y_train, n_splits=3),
    scoring='roc_auc_ovr_weighted',
    refit=True,
    verbose=3,
    n_jobs=-1,
    error_score='raise'
)

search.fit(X_train, y_train)

print(f"Hyperparameter tuning completed in {time.time()-start_time:.2f} seconds")
print(f"Best parameters: {search.best_params_}")
print(f"Best CV score: {search.best_score_:.4f}")

# 6. Evaluate on test set
print("\nEvaluating on test set...")
y_pred = search.predict(X_test)
y_proba = search.predict_proba(X_test)

# Classification report for each disease
diseases = ['oestrus', 'calving', 'lameness', 'mastitis']
test_results = {}

for i, disease in enumerate(diseases):
    print(f"\n{'='*50}\n{disease.upper()} PERFORMANCE\n{'='*50}")
    print(classification_report(y_test[disease], y_pred[:, i]))
    test_results[disease] = {
        'auc': roc_auc_score(y_test[disease], y_proba[i][:, 1])
    }

# 7. Feature Importance Analysis
print("\nFeature Importances:")
best_model = search.best_estimator_.named_steps['clf']
feature_importances = []

for idx, disease in enumerate(diseases):
    importances = best_model.estimators_[idx].feature_importances_
    feature_importances.append(importances)

importance_df = pd.DataFrame(
    np.array(feature_importances).T,
    columns=diseases,
    index=X.columns
)
importance_df['Average'] = importance_df.mean(axis=1)
importance_df = importance_df.sort_values('Average', ascending=False)

print("\nTop Features:")
print(importance_df)

# 8. Save the best model and results
joblib.dump(search.best_estimator_, 'random_forest_multioutput_model.joblib')
importance_df.to_csv('feature_importances.csv')
print("Model and feature importances saved")

# 9. Final summary
print("\n" + "="*50)
print("FINAL MODEL SUMMARY")
print("="*50)
print(f"Best parameters: {search.best_params_}")
print(f"Cross-validation AUC: {search.best_score_:.4f}")

for disease in diseases:
    print(f"{disease.upper()} Test AUC: {test_results[disease]['auc']:.4f}")

Loading and preparing data...
Train shape: (26304, 9), Test shape: (6576, 9)
Features used: ['cow.1', 'hour', 'IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'rest_to_eat_ratio', 'hour_sin', 'hour_cos']
Starting hyperparameter tuning with GridSearchCV...
Fitting 3 folds for each of 5 candidates, totalling 15 fits
Hyperparameter tuning completed in 59.59 seconds
Best parameters: {'clf__estimator__bootstrap': True, 'clf__estimator__max_depth': None, 'clf__estimator__max_features': 0.5, 'clf__estimator__max_samples': 0.6, 'clf__estimator__min_samples_leaf': 1, 'clf__estimator__min_samples_split': 5, 'clf__estimator__n_estimators': 200}
Best CV score: 0.7915

Evaluating on test set...

OESTRUS PERFORMANCE
              precision    recall  f1-score   support

           0       0.82      0.92      0.87      4987
           1       0.58      0.35      0.44      1589

    accuracy                           0.78      6576
   macro avg       0.70      0.64      0.65      6576
weighted avg       